# Notebook 5 — Hard-negative robustness evaluation (not yet run)

## Project
IsoColBERT: Token-Level Isotropy Regularization for Multilingual Late-Interaction Retrieval
Minh Tran, Isaac Chang, Ingrid Chien — Harvard University

## What this notebook does
Evaluation framework for re-scoring saved checkpoints against LaBSE-mined top-20 hard negatives per query, on the four low-resource pairs (SW, AR, NE, TA). Provided as a complete, runnable framework for future work; results do not appear in the paper (the original Hard-Negative Robustness section was dropped during the cleanup pass).

**Status:** NOT YET RUN. Notebook contains no execution outputs. Approximately 30–40 min on a Colab A100 if run.

## Outputs produced (or designed to produce)
- Per-language hard-negative P@1 (vanilla and IsoColBERT)
- Side-by-side comparison of full-pool vs. hard-neg gain shrinkage

## Where this feeds into the paper
- Future work / supplementary robustness check (paper does not currently cite this).

## Reproducibility
- **Model**: XLM-RoBERTa-base + 128-dim linear projection (ColBERT-style late interaction)
- **Training data**: OPUS-100 English-Spanish parallel split, streamed from HuggingFace
- **Evaluation**: FLORES+ dev + devtest combined (≈2009 candidates per language)
- **Seeds**: {42, 1337, 2024}
- **Hardware**: Single NVIDIA A100 (Colab Pro). ~1 GPU-hour per training run.
- **Drive layout**: checkpoints save to `/MyDrive/iso_colbert_lam0p5/`, `/MyDrive/iso_colbert_active_token/`, `/MyDrive/iso_colbert_crossattn/`.

---


# IsoColBERT Hard-Negative Evaluation (2000-step weights)

**Goal.** Re-evaluate the 6 saved checkpoints (3 seeds × {vanilla, iso_lam0p5}) against LaBSE-mined hard negatives instead of the full FLORES+ pool. Tests whether the IsoColBERT retrieval gain on low-resource languages survives a stricter distractor set.

**Setup.** Eval-only, no retraining. Weights are at `/content/drive/MyDrive/iso_colbert_lam0p5/`. For each EN query, we mine the top-20 most-similar candidates from the FLORES+ pool using LaBSE, then score the query against `{true match + 20 hard negatives} = 21 candidates`. Languages: SW, AR, NE, TA.

**Compute.** ~30–40 min on a Colab A100.

In [ ]:
!pip install -q transformers datasets sentence-transformers torch

In [ ]:
import os, json, gc, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import XLMRobertaModel, XLMRobertaTokenizerFast
from datasets import load_dataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'GPU required — Runtime → Change runtime type → GPU'

DIM     = 128
MAX_LEN = 64
K_HARD  = 20             # number of hard negatives per query
SEEDS   = [42, 1337, 2024]
METHODS = ['vanilla', 'iso_lam0p5']
WEIGHTS_DIR = '/content/drive/MyDrive/iso_colbert_lam0p5'
SAVE_DIR    = '/content/drive/MyDrive/iso_colbert_lam0p5'

# Low-resource pairs to evaluate (matches the new main table)
LANGS = [
    ('sw', 'swh_Latn', 'EN-SW'),
    ('ar', 'arb_Arab', 'EN-AR'),
    ('ne', 'npi_Deva', 'EN-NE'),
    ('ta', 'tam_Taml', 'EN-TA'),
]

print(f'Device: {DEVICE}')
print(f'Hard negatives per query: {K_HARD}')
print(f'Seeds: {SEEDS}')
print(f'Languages: {[name for _, _, name in LANGS]}')

In [ ]:
# Mount Drive and check that checkpoints exist
from google.colab import drive
drive.mount('/content/drive')

def ckpt_path(method, seed):
    # Matches the naming used by iso_colbert_lam0p5_multiseed (1).ipynb
    return os.path.join(WEIGHTS_DIR, f'{method}_lam0.5_seed{seed}.pt')

print('Checking saved checkpoints:')
for m in METHODS:
    for s in SEEDS:
        p = ckpt_path(m, s)
        print(f'  {m:12s} seed {s}: {os.path.exists(p)}  |  {p}')

In [ ]:
# HF auth for FLORES+ (gated)
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF token loaded from Colab userdata.')
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()

In [ ]:
# Tokenizer + ColBERT model class (identical to iso_colbert_lam0p5_multiseed)
tokenizer = XLMRobertaTokenizerFast.from_pretrained('xlm-roberta-base')

class ColBERTEncoder(nn.Module):
    def __init__(self, dim=DIM):
        super().__init__()
        self.encoder = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        self.linear  = nn.Linear(768, dim)
        self.dim     = dim

    def encode(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        token_vecs = self.linear(out.last_hidden_state)
        token_vecs = F.normalize(token_vecs, dim=-1)
        return token_vecs, attention_mask.bool()

    def forward(self, input_ids, attention_mask):
        return self.encode(input_ids, attention_mask)

print('Tokenizer + ColBERTEncoder ready.')

In [ ]:
# Load FLORES+ dev+devtest combined for each language (matches the main retrieval pool)
def combined(lang_code):
    dev = load_dataset('openlanguagedata/flores_plus', lang_code, split='dev')
    devtest = load_dataset('openlanguagedata/flores_plus', lang_code, split='devtest')
    return [r['text'] for r in dev] + [r['text'] for r in devtest]

print('Loading EN pool (eng_Latn)...')
EN_POOL = combined('eng_Latn')
print(f'  EN pool size: {len(EN_POOL)}')

TGT_POOLS = {}
for short, code, name in LANGS:
    print(f'Loading {name} pool ({code})...')
    TGT_POOLS[short] = combined(code)
    print(f'  {name} pool size: {len(TGT_POOLS[short])}')

# Sanity: every EN[i] is the parallel translation of TGT[i] in FLORES+
for short, _, name in LANGS:
    assert len(EN_POOL) == len(TGT_POOLS[short]), f'Pool size mismatch on {name}'

In [ ]:
# LaBSE mining — done ONCE per target language, cached to disk
from sentence_transformers import SentenceTransformer

LABSE_CACHE = os.path.join(SAVE_DIR, 'labse_hardneg_indices.json')

def mine_hard_negatives(en_texts, tgt_texts, k=K_HARD):
    """For each EN query i, return top-k indices in tgt_texts ranked by LaBSE similarity,
    EXCLUDING the true match index i."""
    labse = SentenceTransformer('sentence-transformers/LaBSE', device=DEVICE)
    labse.eval()
    en_emb  = labse.encode(en_texts,  batch_size=64, convert_to_tensor=True,
                           normalize_embeddings=True, show_progress_bar=True)
    tgt_emb = labse.encode(tgt_texts, batch_size=64, convert_to_tensor=True,
                           normalize_embeddings=True, show_progress_bar=True)
    # Similarity matrix — EN × TGT
    sim = en_emb @ tgt_emb.T
    N   = sim.size(0)
    # Mask out the diagonal (true matches) before taking top-k
    diag_mask = torch.eye(N, dtype=torch.bool, device=sim.device)
    sim_masked = sim.masked_fill(diag_mask, float('-inf'))
    topk = torch.topk(sim_masked, k=k, dim=-1).indices  # (N, k)
    del labse, en_emb, tgt_emb, sim, sim_masked
    gc.collect(); torch.cuda.empty_cache()
    return topk.cpu().tolist()

# Load cache if available, otherwise mine
if os.path.exists(LABSE_CACHE):
    with open(LABSE_CACHE) as f:
        HARD_IDX = json.load(f)
    print(f'Loaded LaBSE-mined hard negatives from {LABSE_CACHE}')
    print(f'  cached languages: {list(HARD_IDX.keys())}')
else:
    HARD_IDX = {}

for short, _, name in LANGS:
    if short in HARD_IDX:
        print(f'  [skip] {name} hard negatives already cached')
        continue
    print(f'Mining {name} hard negatives with LaBSE...')
    HARD_IDX[short] = mine_hard_negatives(EN_POOL, TGT_POOLS[short], k=K_HARD)
    with open(LABSE_CACHE, 'w') as f:
        json.dump(HARD_IDX, f)
    print(f'  done. saved to {LABSE_CACHE}')

# Sanity: 20 distinct indices per query, none equal to the diagonal
for short, _, name in LANGS:
    arr = np.array(HARD_IDX[short])
    assert arr.shape == (len(EN_POOL), K_HARD), f'{name} shape: {arr.shape}'
    diag_violations = sum(i in arr[i] for i in range(len(EN_POOL)))
    print(f'  {name}: shape OK, diagonal violations = {diag_violations}')

In [ ]:
# ColBERT encoding + hard-negative scoring
@torch.no_grad()
def encode_all_padded(model, texts, max_len=MAX_LEN, batch_size=64):
    model.eval()
    raw_v, raw_m = [], []
    L_max = 0
    for i in range(0, len(texts), batch_size):
        tok = tokenizer(texts[i:i+batch_size], padding=True, truncation=True,
                        max_length=max_len, return_tensors='pt')
        tok = {k: v.to(DEVICE) for k, v in tok.items()}
        v, m = model.encode(tok['input_ids'], tok['attention_mask'])
        raw_v.append(v); raw_m.append(m)
        L_max = max(L_max, v.size(1))
    padded_v, padded_m = [], []
    for v, m in zip(raw_v, raw_m):
        if v.size(1) < L_max:
            pad_v = torch.zeros(v.size(0), L_max - v.size(1), v.size(2), device=v.device)
            pad_m = torch.zeros(m.size(0), L_max - m.size(1), dtype=m.dtype, device=m.device)
            v = torch.cat([v, pad_v], dim=1)
            m = torch.cat([m, pad_m], dim=1)
        padded_v.append(v); padded_m.append(m)
    return torch.cat(padded_v, 0), torch.cat(padded_m, 0)


@torch.no_grad()
def maxsim_score_pair(qv, qm, dv, dm):
    """Single query-document pair MaxSim. qv: (Lq, D), dv: (Ld, D)."""
    sim = qv @ dv.T                                # (Lq, Ld)
    sim = sim.masked_fill(~dm[None, :], -1e9)
    maxsim = sim.max(dim=-1).values * qm.float()   # (Lq,)
    return maxsim.sum().item()


@torch.no_grad()
def hardneg_p_at_1(model, en_texts, tgt_texts, hard_indices):
    """For each query i, score against {tgt_texts[i]} ∪ {tgt_texts[j] for j in hard_indices[i]}.
    Return P@1 = fraction of queries where the true match (index 0 in the candidate set) wins."""
    q_v, q_m = encode_all_padded(model, en_texts)
    d_v, d_m = encode_all_padded(model, tgt_texts)
    N = q_v.size(0)
    correct = 0
    for i in range(N):
        cand_ids = [i] + list(hard_indices[i])     # true match first, then K hard negs
        scores = [maxsim_score_pair(q_v[i], q_m[i], d_v[j], d_m[j]) for j in cand_ids]
        if int(np.argmax(scores)) == 0:
            correct += 1
    return correct / N

print('Eval functions ready.')

In [ ]:
# Run the eval loop — 6 models × 4 languages = 24 evaluations
HARDNEG_PATH = os.path.join(SAVE_DIR, 'hardneg_metrics_lam0.5.json')

if os.path.exists(HARDNEG_PATH):
    with open(HARDNEG_PATH) as f:
        hardneg_metrics = json.load(f)
    print(f'Resuming from {HARDNEG_PATH}')
else:
    hardneg_metrics = {}

for method in METHODS:
    for seed in SEEDS:
        key = f'{method}_seed{seed}'
        if key in hardneg_metrics and all(f'p1_{s}_hn' in hardneg_metrics[key] for s, _, _ in LANGS):
            print(f'[skip] {key} already evaluated.')
            continue

        print(f'\n{"="*80}\nEVAL: {key}\n{"="*80}')
        ckpt = ckpt_path(method, seed)
        model = ColBERTEncoder().to(DEVICE)
        sd = torch.load(ckpt, map_location=DEVICE)
        if isinstance(sd, dict) and 'state_dict' in sd:
            sd = sd['state_dict']
        model.load_state_dict(sd, strict=False)
        model.eval()

        results = hardneg_metrics.get(key, {})
        for short, _, name in LANGS:
            metric_key = f'p1_{short}_hn'
            if metric_key in results:
                print(f'  [skip] {name} already done: {results[metric_key]:.4f}')
                continue
            print(f'  Evaluating {name} hard negatives (K={K_HARD})...')
            p1_hn = hardneg_p_at_1(model, EN_POOL, TGT_POOLS[short], HARD_IDX[short])
            results[metric_key] = p1_hn
            print(f'    {name} P@1 (hard-neg) = {p1_hn:.4f}')
            hardneg_metrics[key] = results
            with open(HARDNEG_PATH, 'w') as f:
                json.dump(hardneg_metrics, f, indent=2)

        del model
        gc.collect(); torch.cuda.empty_cache()

print(f'\n✅ All hard-negative evaluations done. Results in {HARDNEG_PATH}')

In [ ]:
# Aggregate — 3-seed mean ± std per language, with significance markers
def agg(method, metric_key):
    vals = [hardneg_metrics[f'{method}_seed{s}'][metric_key] for s in SEEDS]
    return float(np.mean(vals)), float(np.std(vals)), vals

def sig_marker(delta_mu, std_iso):
    if std_iso == 0: return ''
    if delta_mu >  2 * std_iso: return '***'
    if delta_mu >      std_iso: return '** '
    if delta_mu >  0.5*std_iso: return '*  '
    return '   '

print('=' * 110)
print(f'HARD-NEGATIVE RESULTS  (K={K_HARD}, LaBSE-mined, λ=0.5, 3 seeds)')
print('=' * 110)
print(f'{"Language pair":<14}  {"Vanilla (μ±σ)":<22}  {"IsoColBERT (μ±σ)":<22}  {"Δ μ":>9}  Sig')
print('-' * 110)

summary = []
for short, _, name in LANGS:
    key = f'p1_{short}_hn'
    v_mu, v_sd, v_vals = agg('vanilla', key)
    i_mu, i_sd, i_vals = agg('iso_lam0p5', key)
    d_mu = i_mu - v_mu
    sig = sig_marker(d_mu, i_sd)
    print(f'{name:<14}  {v_mu:.4f} ± {v_sd:.4f}    {i_mu:.4f} ± {i_sd:.4f}    {d_mu:+.4f}  {sig}')
    summary.append((name, v_mu, v_sd, i_mu, i_sd, d_mu, sig, v_vals, i_vals))

print()
print('Significance: *** Δμ > 2σ  |  ** Δμ > σ  |  * Δμ > σ/2')
print()
print('--- Per-seed breakdown ---')
for name, _, _, _, _, _, _, v_vals, i_vals in summary:
    print(f'\n{name}:')
    for s, vv, ii in zip(SEEDS, v_vals, i_vals):
        print(f'  seed {s:>4}: vanilla={vv:.4f}  iso={ii:.4f}  Δ={ii - vv:+.4f}')

In [ ]:
# Compare hard-neg results to the full-pool results from iso_colbert_lam0p5
# (so you can write a one-line robustness paragraph for the paper)
FULL_POOL_PATH = os.path.join(SAVE_DIR, 'metrics_lam0.5_with_low_resource.json')
if not os.path.exists(FULL_POOL_PATH):
    FULL_POOL_PATH = os.path.join(SAVE_DIR, 'metrics_lam0.5.json')

try:
    with open(FULL_POOL_PATH) as f:
        full_pool = json.load(f)

    print('=' * 90)
    print('FULL POOL  vs  HARD-NEG POOL  (IsoColBERT − Vanilla, 3-seed mean)')
    print('=' * 90)
    print(f'{"Language pair":<14}  {"Full-pool Δ":>12}  {"Hard-neg Δ":>13}  {"Shrinkage":>11}')
    print('-' * 90)
    for short, _, name in LANGS:
        fk = f'p1_{short}'
        full_v = np.mean([full_pool[f'vanilla_seed{s}'][fk]    for s in SEEDS])
        full_i = np.mean([full_pool[f'iso_lam0p5_seed{s}'][fk] for s in SEEDS])
        full_d = full_i - full_v
        hn_v = np.mean([hardneg_metrics[f'vanilla_seed{s}'][f'p1_{short}_hn']    for s in SEEDS])
        hn_i = np.mean([hardneg_metrics[f'iso_lam0p5_seed{s}'][f'p1_{short}_hn'] for s in SEEDS])
        hn_d = hn_i - hn_v
        shrink = hn_d - full_d
        print(f'{name:<14}  {full_d:+12.4f}  {hn_d:+13.4f}  {shrink:+11.4f}')
except FileNotFoundError:
    print(f'Full-pool metrics not found at {FULL_POOL_PATH}.')
    print('Skipping comparison — just use the hard-neg table above.')